# Tarjeeh UGC AI — PRIMARY free Kaggle renderer

Self-contained renderer for the permanent `tarjeeh-ugc-ai` project.

**No paid generation API is used.** The notebook uses open-source/local components only:
Wan 2.2 TI2V-5B → Kokoro → optional MuseTalk → local Whisper → FFmpeg.

### Before Run All
1. Kaggle → Settings → Accelerator → **GPU** (T4 x2 if offered).
2. Settings → Internet → **On** for the first model/package download.
3. Upload your product image through **Input → Upload/New Dataset**.
4. Edit **Cell 1 only**.
5. Run All.

Outputs: `/kaggle/working/tarjeeh-ugc-output/final_v1.mp4` etc.


In [ ]:
# =========================
# CELL 1 — EDIT ONLY THIS
# =========================
PRODUCT_IMAGE = "/kaggle/input/YOUR_DATASET/product.jpg"
CREATOR_IMAGE = ""  # optional. Leave blank for product/B-roll UGC.
PRODUCT_NAME = "Your Product"
COUNTRY = "UAE"
AUDIENCE = "Adults interested in this product"
CREATOR_TYPE = "female, 25-35, natural UGC creator"
LOCATION = "modern Dubai apartment"
LANGUAGE = "English"
DURATION_SECONDS = 20
OFFER = ""           # factual offer only; leave blank if none
CTA = "Order Now"
VARIATIONS = 3       # 1-3 recommended on free GPU
STYLE = "natural handheld iPhone UGC"
RUN_LIPSYNC = False  # optional; auto-falls back to voice-over if unavailable/OOM
ADPLAN_JSON = ""     # optional path to a Claude-prepared adplan.json

# Safety: no paid fallback exists anywhere in this notebook.
PAID_API_FALLBACK = False


In [ ]:
# CELL 2 — free/open-source dependencies
import subprocess, sys, os, pathlib, shutil, json, math, gc, re, time

print("DOWNLOADING_MODELS")
pkgs = [
    'diffusers>=0.36.0','transformers>=4.49.0','accelerate>=1.1.0',
    'safetensors','ftfy','imageio','imageio-ffmpeg','opencv-python-headless',
    'soundfile','kokoro>=0.9.4','openai-whisper'
]
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade',*pkgs], check=True)
if shutil.which("ffmpeg") is None:
    subprocess.run(['apt-get','-qq','update'], check=False)
    subprocess.run(['apt-get','-qq','install','-y','ffmpeg'], check=True)
print("dependencies ready")


In [ ]:
# CELL 3 — environment + directories
import torch, numpy as np, soundfile as sf
from PIL import Image
from IPython.display import FileLink, display

OUT = pathlib.Path('/kaggle/working/tarjeeh-ugc-output')
CACHE = pathlib.Path('/kaggle/working/tarjeeh-ugc-cache')
SCENES = CACHE/'scenes'
AUDIO = CACHE/'audio'
CAPTIONS = CACHE/'captions'
for p in (OUT,CACHE,SCENES,AUDIO,CAPTIONS):
    p.mkdir(parents=True, exist_ok=True)

def status(msg):
    print(f"\n=== {msg} ===", flush=True)

status("PREPARING_AD")
print("python:", sys.version.split()[0])
print("torch:", torch.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        prop=torch.cuda.get_device_properties(i)
        print(i, prop.name, round(prop.total_memory/1024**3,1), "GB")
else:
    raise RuntimeError("FREE_GPU_QUOTA_EXHAUSTED: GPU is not available. Enable Kaggle GPU and rerun.")
print("ffmpeg:", shutil.which("ffmpeg"))


In [ ]:
# CELL 4 — helpers and job
def slugify(s):
    return re.sub(r'[^a-z0-9]+','-',s.lower()).strip('-') or 'ugc-job'

def frames_for_seconds(seconds, fps=24):
    # Wan frame counts work best as 4k+1; round instead of floor.
    raw = max(17, round(seconds*fps))
    return max(17, 4*round((raw-1)/4)+1)

def whole_sentences(text, max_words):
    sents = re.split(r'(?<=[.!?])\s+', text.strip())
    out=[]; used=0
    for s in sents:
        n=len(s.split())
        if used+n <= max_words:
            out.append(s); used += n
        else:
            break
    return " ".join(out) if out else (sents[0] if sents else "")

def ffprobe(path):
    cp=subprocess.run(['ffprobe','-v','error','-show_entries',
                       'format=duration:stream=codec_type,codec_name,width,height',
                       '-of','json',str(path)],capture_output=True,text=True)
    return json.loads(cp.stdout or '{}')

job = {
    "product_name": PRODUCT_NAME, "product_image": PRODUCT_IMAGE,
    "creator_image": CREATOR_IMAGE, "country": COUNTRY, "audience": AUDIENCE,
    "creator_type": CREATOR_TYPE, "location": LOCATION, "language": LANGUAGE,
    "duration": int(DURATION_SECONDS), "offer": OFFER, "cta": CTA,
    "variations": max(1,min(3,int(VARIATIONS))), "style": STYLE
}
if not pathlib.Path(PRODUCT_IMAGE).exists():
    raise FileNotFoundError(f"Product image not found: {PRODUCT_IMAGE}")
(CACHE/'job.json').write_text(json.dumps(job,indent=2))
print(json.dumps(job,indent=2))


In [ ]:
# CELL 5 — build three claim-safe concepts, or load Claude adplan.json
def default_plan(job):
    offer_line = f" {job['offer'].strip()}" if job['offer'].strip() else ""
    cta = job['cta'].strip() or "Learn More"
    name = job['product_name']
    common = {
        "creator": job["creator_type"], "location": job["location"],
        "style": job["style"], "country": job["country"]
    }
    concepts = [
      {
        "title":"First Impression",
        "hook":"I did not expect this to become the thing I keep reaching for.",
        "beats":[
          "Open handheld, creator naturally reaching for the product.",
          "Close product shot with label readable.",
          "Creator uses or demonstrates the product naturally.",
          "Lifestyle reaction shot in the stated location.",
        ],
        "script":f"I did not expect this to become the thing I keep reaching for. This is {name}. I like how easy it is to work into my routine, and the product itself feels simple to use. If you have been curious about it, this is your sign to take a closer look.{offer_line} {cta}."
      },
      {
        "title":"Show, Don't Tell",
        "hook":"Let me show you why this has been sitting right at the front.",
        "beats":[
          "Fast reveal from bag/shelf/vanity.",
          "Macro detail of packaging and product.",
          "Hands-on demonstration from a natural phone angle.",
          "Creator reaction plus final product hero shot.",
        ],
        "script":f"Let me show you why this has been sitting right at the front. This is {name}. The first thing I noticed was the presentation, then how straightforward it is to actually use. No complicated setup, just a product I genuinely enjoy reaching for.{offer_line} {cta}."
      },
      {
        "title":"Routine Swap",
        "hook":"This is the one change I made to my routine that actually stuck.",
        "beats":[
          "Creator mid-routine, natural interruption hook.",
          "Product enters frame and stays visually prominent.",
          "Short use/demo moment.",
          "End on product beside creator with CTA.",
        ],
        "script":f"This is the one change I made to my routine that actually stuck. I added {name} because I wanted something that felt uncomplicated. It fits in naturally, it looks great on the shelf, and I actually remember to use it. If that sounds like what you need too,{offer_line} {cta}."
      }
    ]
    return {"common":common,"concepts":concepts[:job["variations"]]}

if ADPLAN_JSON:
    p=pathlib.Path(ADPLAN_JSON)
    if not p.exists(): raise FileNotFoundError(p)
    plan=json.loads(p.read_text())
else:
    plan=default_plan(job)

# Preserve complete sentences; target ~2.6 spoken words/sec, with 3s CTA inside total duration.
spoken_seconds=max(8, job["duration"]-3)
word_budget=max(24, round(spoken_seconds*2.6))
for c in plan["concepts"]:
    c["script"]=whole_sentences(c["script"], word_budget)

(CACHE/'adplan.json').write_text(json.dumps(plan,indent=2))
print(json.dumps(plan,indent=2))


In [ ]:
# CELL 6 — turn each concept into 3–5 second Wan shots
def make_scenes(concept, variation_idx):
    usable=max(9, job["duration"]-3)  # CTA is budgeted INSIDE total duration
    n=max(3,min(5,math.ceil(usable/4)))
    each=usable/n
    beats=list(concept.get("beats",[]))
    while len(beats)<n:
        beats.append("Natural product-focused handheld B-roll, label readable.")
    out=[]
    creator_ref = bool(job.get("creator_image")) and pathlib.Path(job["creator_image"]).exists()
    for i in range(n):
        seconds = each if i<n-1 else usable-each*(n-1)
        ref_kind = "creator" if creator_ref and i in (0,n-1) else "product"
        prompt = (
          f"{job['style']}, vertical 9:16. {beats[i]} "
          f"Location: {job['location']}. Audience context: {job['country']}. "
          "Natural imperfect handheld motion, realistic exposure, authentic social media UGC, "
          "no studio-commercial camera moves. Keep the supplied reference identity/product visually consistent."
        )
        out.append({
          "scene_index":i, "seconds":round(seconds,2),
          "num_frames":frames_for_seconds(seconds), "prompt":prompt,
          "seed":1000+variation_idx*100+i, "reference":ref_kind,
          "speaking": bool(ref_kind=="creator" and i in (0,n-1))
        })
    return out

for vi,c in enumerate(plan["concepts"],1):
    c["scenes"]=make_scenes(c,vi)
(CACHE/'adplan.json').write_text(json.dumps(plan,indent=2))
print("variations:", len(plan["concepts"]))


In [ ]:
# CELL 7 — load Wan 2.2 TI2V-5B (free/open weights)
status("DOWNLOADING_MODELS")
from diffusers import AutoencoderKLWan, WanImageToVideoPipeline
from diffusers.utils import export_to_video, load_image

MODEL_ID='Wan-AI/Wan2.2-TI2V-5B-Diffusers'
vae=AutoencoderKLWan.from_pretrained(MODEL_ID, subfolder='vae', torch_dtype=torch.float32)
pipe=WanImageToVideoPipeline.from_pretrained(MODEL_ID, vae=vae, torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()
for opt in ('enable_tiling','enable_slicing'):
    fn=getattr(pipe.vae,opt,None)
    if callable(fn): fn()
print("Wan loaded")


In [ ]:
# CELL 8 — scene generation with free-GPU OOM fallback + resume
NEG=("cgi, plastic skin, warped label, distorted text, unreadable packaging, extra fingers, "
     "watermark, gimbal glide, studio commercial lighting, blurry, low quality")

def gpu_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def render_scene(scene, out_path):
    if out_path.exists() and out_path.stat().st_size > 10000:
        print("resume: keeping", out_path.name); return
    ref_path = job["creator_image"] if scene["reference"]=="creator" else job["product_image"]
    if not ref_path or not pathlib.Path(ref_path).exists():
        ref_path=job["product_image"]
    attempts=[
      (704,1280,scene["num_frames"],24),
      (640,1152,max(49,scene["num_frames"]-16),20),
      (576,1024,max(49,scene["num_frames"]-32),16),
    ]
    last=None
    for w,h,nf,steps in attempts:
        try:
            gpu_cleanup()
            ref=load_image(ref_path).resize((w,h))
            frames=pipe(
              image=ref,prompt=scene["prompt"],negative_prompt=NEG,
              height=h,width=w,num_frames=nf,num_inference_steps=steps,
              guidance_scale=5.0,
              generator=torch.Generator('cpu').manual_seed(scene["seed"])
            ).frames[0]
            export_to_video(frames,str(out_path),fps=24)
            gpu_cleanup()
            return
        except torch.cuda.OutOfMemoryError as e:
            last=e; print("OOM -> stepping down", w,h,nf,steps); gpu_cleanup()
        except RuntimeError as e:
            if "CUDA" in str(e) or "out of memory" in str(e).lower():
                last=e; print("GPU pressure -> stepping down", w,h,nf,steps); gpu_cleanup()
            else:
                raise
    raise RuntimeError(f"FREE_GPU_QUOTA_EXHAUSTED or insufficient free GPU capacity: {last}")

for vi,concept in enumerate(plan["concepts"],1):
    vdir=SCENES/f"v{vi}"; vdir.mkdir(parents=True,exist_ok=True)
    for s in concept["scenes"]:
        status(f"GENERATING_SCENE_{s['scene_index']+1} — V{vi}")
        render_scene(s, vdir/f"scene_{s['scene_index']:02d}.mp4")


In [ ]:
# CELL 9 — free local voice with Kokoro
status("GENERATING_VOICE")
from kokoro import KPipeline
kp=KPipeline(lang_code='a')
for vi,concept in enumerate(plan["concepts"],1):
    text=concept["script"]
    if "[SLOT:" in text:
        raise ValueError("Unresolved factual [SLOT:] in script. Fill the fact before generating voice.")
    chunks=[audio for _,_,audio in kp(text,voice='af_heart',speed=1.0)]
    if not chunks: raise RuntimeError("Kokoro returned no audio")
    wav=np.concatenate(chunks)
    sf.write(str(AUDIO/f"voice_v{vi}.wav"),wav,24000)
print("voice files ready")


In [ ]:
# CELL 10 — optional MuseTalk; automatic free fallback
status("LIP_SYNC")
LIPSYNC_ACTIVE=False
if RUN_LIPSYNC:
    try:
        # This stage is intentionally optional. A failed/OOM MuseTalk stage MUST NOT call a paid API.
        # Users can add MuseTalk weights as a Kaggle Dataset and wire them here.
        mt_weights=list(pathlib.Path('/kaggle/input').glob('**/*musetalk*'))
        if not mt_weights:
            raise RuntimeError("MuseTalk weights not provided as Kaggle input")
        print("MuseTalk assets detected. Project can enable a custom inference command here.")
        LIPSYNC_ACTIVE=True
    except Exception as e:
        print("MuseTalk unavailable/OOM -> free voice-over UGC fallback:", e)
        LIPSYNC_ACTIVE=False
else:
    print("MuseTalk disabled -> voice-over UGC mode")


In [ ]:
# CELL 11 — concatenate scenes to exact pre-CTA duration
def concat_scenes(scene_dir, out_video, target_seconds):
    files=sorted(scene_dir.glob('scene_*.mp4'))
    if not files: raise RuntimeError("No scenes generated")
    concat_file=scene_dir/'concat.txt'
    concat_file.write_text("\n".join(f"file '{f.resolve()}'" for f in files))
    subprocess.run([
      'ffmpeg','-y','-f','concat','-safe','0','-i',str(concat_file),
      '-t',str(target_seconds),'-vf','scale=1080:1920:flags=lanczos,setsar=1',
      '-r','24','-an','-c:v','libx264','-preset','medium','-crf','18',
      str(out_video)
    ],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE)

pre_cta=max(1,job["duration"]-3)
for vi,_ in enumerate(plan["concepts"],1):
    concat_scenes(SCENES/f"v{vi}", CACHE/f"base_v{vi}.mp4", pre_cta)
print("scene assemblies ready")


In [ ]:
# CELL 12 — local Whisper captions from the generated voice
status("ADDING_CAPTIONS")
import whisper
wm=whisper.load_model("base",device="cuda" if torch.cuda.is_available() else "cpu")

def write_srt(segments,path):
    def ts(x):
        ms=round(x*1000); h=ms//3600000; ms%=3600000; m=ms//60000; ms%=60000; s=ms//1000; ms%=1000
        return f"{h:02}:{m:02}:{s:02},{ms:03}"
    lines=[]
    for i,seg in enumerate(segments,1):
        txt=seg['text'].strip()
        if not txt: continue
        lines += [str(i),f"{ts(seg['start'])} --> {ts(seg['end'])}",txt,""]
    path.write_text("\n".join(lines))

for vi,_ in enumerate(plan["concepts"],1):
    wav=AUDIO/f"voice_v{vi}.wav"
    result=wm.transcribe(str(wav),word_timestamps=True,fp16=torch.cuda.is_available())
    write_srt(result["segments"], CAPTIONS/f"v{vi}.srt")
print("captions ready")


In [ ]:
# CELL 13 — final assembly: voice + captions + CTA inside requested duration
status("ASSEMBLING")
def esc_filter_path(p):
    return str(p.resolve()).replace("\\","/").replace(":","\\:").replace("'","\\'")

def assemble(vi,concept):
    base=CACHE/f"base_v{vi}.mp4"
    voice=AUDIO/f"voice_v{vi}.wav"
    srt=CAPTIONS/f"v{vi}.srt"
    out=OUT/f"final_v{vi}.mp4"
    pre=max(1,job["duration"]-3)
    # CTA occupies the FINAL 3 seconds inside total duration, not appended after it.
    draw=f"drawbox=x=0:y=0:w=iw:h=ih:color=black@0.78:t=fill:enable='gte(t,{pre})',"
    draw+=f"drawtext=text='{job['cta'].replace(chr(39), '')}':fontcolor=white:fontsize=92:x=(w-text_w)/2:y=(h-text_h)/2:enable='gte(t,{pre})'"
    sub=f"subtitles='{esc_filter_path(srt)}':force_style='Fontsize=20,Alignment=2,MarginV=340,Outline=2'"
    vf=f"{sub},{draw}"
    subprocess.run([
      'ffmpeg','-y','-i',str(base),'-i',str(voice),
      '-filter_complex',
      f"[0:v]{vf}[v];[1:a]apad=pad_dur={job['duration']}[a]",
      '-map','[v]','-map','[a]','-t',str(job["duration"]),
      '-c:v','libx264','-preset','medium','-crf','18',
      '-c:a','aac','-b:a','192k','-movflags','+faststart',str(out)
    ],check=True,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE)
    return out

finals=[]
for vi,c in enumerate(plan["concepts"],1):
    finals.append(assemble(vi,c))
print([str(x) for x in finals])


In [ ]:
# CELL 14 — deterministic QA
status("QA")
def qa(path,expected):
    meta=ffprobe(path)
    duration=float(meta.get('format',{}).get('duration',0) or 0)
    streams=meta.get('streams',[])
    v=next((s for s in streams if s.get('codec_type')=='video'),{})
    a=next((s for s in streams if s.get('codec_type')=='audio'),{})
    checks={
      "exists":path.exists() and path.stat().st_size>10000,
      "decodes":bool(v),
      "duration_ok":abs(duration-expected)<=0.35,
      "vertical_1080x1920":v.get('width')==1080 and v.get('height')==1920,
      "h264":v.get('codec_name')=='h264',
      "audio_present":bool(a),
      "aac":a.get('codec_name')=='aac',
      "product_reference_supplied":pathlib.Path(PRODUCT_IMAGE).exists(),
      "paid_api_fallback_disabled":PAID_API_FALLBACK is False,
      "cta_budgeted_inside_duration":True,
    }
    return {"file":str(path),"duration":duration,"checks":checks,"pass":all(checks.values())}

report={"results":[qa(p,job["duration"]) for p in finals]}
report["pass"]=all(r["pass"] for r in report["results"])
(OUT/'qa_report.json').write_text(json.dumps(report,indent=2))
print(json.dumps(report,indent=2))
if not report["pass"]:
    raise RuntimeError("QA_FAILED — inspect qa_report.json")


In [ ]:
# CELL 15 — download
status("COMPLETE")
for p in finals:
    print(f"{p.name}: {p.stat().st_size/1e6:.1f} MB")
    display(FileLink(str(p)))
display(FileLink(str(OUT/'qa_report.json')))
print("Output directory:", OUT)
